# Rush yards feature lab

Use this notebook to see how the rushing-yards model performs and which features matter, then try adding and dropping features.

**Nothing here writes to the database.** It reads `nfl.*` and `model.*` in read-only transactions and refits LightGBM in memory. The helpers live in `backtest/rush_yards_lab.py`.

**Protocol (fixed up front so tuning can't fool us):**
- **Validation seasons 2019–2024.** Each fold trains on every earlier season and predicts one whole season. That's one refit per season instead of per week, so each experiment takes a few seconds. Numbers come out slightly worse than the weekly production backtest, but comparisons between feature lists are fair.
- **Five seeds, averaged.** With a single seed, just reordering the same columns moves MAE by about 0.02 yards, enough to fake a feature effect. Every fit is repeated over `lab.SEEDS` and the predictions averaged.
- **2025 is the holdout.** `evaluate` refuses to score it unless you pass `allow_holdout=True`. Look at it once, after your feature list is final (section 8).
- **Compare on the same rows, with error bars.** `compare` reports the paired MAE change and a standard error clustered by week. Gains here are a fraction of a yard, so treat anything under about 2 standard errors, or under about 0.02 yards, as noise.

Run from the repo root with `jupyter notebook backtest/rush_yards_lab.ipynb`. The first cell points the connection at Supabase.

In [ ]:
import os

import matplotlib.pyplot as plt
import polars as pl
from sqlalchemy import create_engine, text

from backtest import rush_yards_lab as lab
from backtest.rush_yards_report import list_runs, load_report
from common.env import load_dotenv
from common.settings import get_settings

load_dotenv()
if os.environ.get("SUPABASE_URL"):
    os.environ["DATABASE_URL"] = os.environ["SUPABASE_URL"]
    os.environ["APP_ENV"] = "production"
engine = create_engine(get_settings().sqlalchemy_url)

pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(40)
pl.Config.set_float_precision(3)
plt.rcParams["figure.figsize"] = (9, 4)


def barh(df, label, value, title, err=None):
    df = df.sort(value)
    fig, ax = plt.subplots(figsize=(9, max(3, 0.3 * df.height)))
    ax.barh(
        df[label].to_list(),
        df[value].to_list(),
        xerr=None if err is None else (2 * df[err]).to_list(),
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## 1. Stored backtest (the official numbers)

These come from the weekly walk-forward runs written by `python -m models.rush_yards backtest`. They're read straight from `model.metric`; nothing is refit.

In [ ]:
with engine.connect() as conn:
    conn.execute(text("SET TRANSACTION READ ONLY"))
    runs = [run for run in list_runs(conn) if run.has_metrics]
    newest = {}
    for run in runs:
        newest.setdefault(run.model_versions, run.run_id)
    report = load_report(conn, list(newest.values()))

stored = pl.DataFrame([m.model_dump() for m in report.metrics]).filter(
    pl.col("season").is_not_null()
)
fig, ax = plt.subplots()
for version in sorted(set(stored["model_version"])):
    part = stored.filter(pl.col("model_version") == version).sort("season")
    ax.plot(part["season"], part["mae"], marker="o", label=version)
ax.set_ylabel("MAE (yards)")
ax.set_title("Stored walk-forward MAE by season")
ax.legend(fontsize=8)
plt.show()
pl.DataFrame([c.model_dump() for c in report.common]).select(
    "model_version", "mae", "rmse", "n"
)

## 2. Load the lab data

One read of `nfl.*`, about 15 seconds. Each row is one regular-season RB game from 2016 to 2025, with every `rush_rb_v1` feature built at that game's kickoff.

In [ ]:
data = lab.load_lab_data(engine)
print(f"{len(data.y):,} labeled RB games; features: {len(data.names)}")

## 3. Reference model: pregame-safe LightGBM

This is the model the experiments are measured against. `scores` puts it next to both baselines (season-to-date and last-4 averages).

In [ ]:
reference = lab.evaluate(data, lab.PREGAME_SAFE)
ref_scores = lab.scores(data, reference)
ref_scores.filter(pl.col("season") == "overall")

In [ ]:
pivot = ref_scores.filter(pl.col("season") != "overall").pivot(
    on="model", index="season", values="mae"
)
fig, ax = plt.subplots()
for model in ("lightgbm", "baseline_std", "baseline_last4"):
    ax.plot(pivot["season"], pivot[model], marker="o", label=model)
ax.set_ylabel("MAE (yards)")
ax.set_title("Lab: validation MAE by season")
ax.legend()
plt.show()
pivot

### Where it misses

- **Calibration:** within each predicted-value decile, does the average prediction match the average actual? A gap at the top end means it over-predicts heavy-use backs.
- **Point in the season:** early weeks have little this-season history.

In [ ]:
cal = lab.calibration(data, reference)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(cal["mean_pred"], cal["mean_actual"], marker="o")
lim = max(cal["mean_pred"].max(), cal["mean_actual"].max()) * 1.05
ax.plot([0, lim], [0, lim], linestyle="--", color="gray")
ax.set_xlabel("mean predicted")
ax.set_ylabel("mean actual")
ax.set_title("Calibration by decile")
plt.show()
display(cal)
lab.error_by_week(data, reference)

## 4. Which features matter

Three views, from cheapest to most trustworthy:

1. **Gain:** LightGBM's built-in importance. Biased toward many-valued features, and it splits credit between correlated ones.
2. **SHAP:** how many yards each feature moves a typical held-out prediction.
3. **Permutation:** how much held-out MAE gets worse when the feature is shuffled. This is the one that answers "is it helping?" Near zero means the model isn't really using it. Negative means it's hurting.

In [ ]:
gain = lab.gain_importance(reference)
shap = lab.shap_importance(data, reference)
perm = lab.permutation_importance(data, reference, repeats=3)

importance = (
    gain.join(shap, on="feature")
    .join(perm, on="feature")
    .rename({"delta_mae": "perm_delta_mae"})
    .sort("perm_delta_mae", descending=True)
)
barh(
    importance,
    "feature",
    "perm_delta_mae",
    "Permutation: MAE increase when shuffled (yards)",
)
barh(importance, "feature", "mean_abs_yards", "SHAP: mean |contribution| (yards)")
importance

## 5. Redundancy and coverage

Feature pairs that are highly correlated carry mostly the same information, so they're candidates for dropping one of the pair. Confirm with the ablation in section 6 before dropping anything.

Coverage shows how often each feature is `Unknown`. A feature that's mostly unknown can't help much.

In [ ]:
display(lab.redundancy(data, lab.PREGAME_SAFE, threshold=0.8))
lab.coverage(data, lab.FULL)

## 6. Ablation: refit without a group

The most trustworthy test. Drop a group, refit every fold, and compare on the same rows. Here `delta_mae > 0` means dropping the group **hurts**, so the group is helping. Error bars are ±2 standard errors.

The second cell drops one feature at a time. It takes about a minute.

In [ ]:
groups = {
    name: cols
    for name, cols in lab.FEATURE_GROUPS.items()
    if set(cols) <= set(lab.PREGAME_SAFE)
}
by_group = lab.ablation(data, lab.PREGAME_SAFE, groups, reference=reference)
barh(
    by_group,
    "dropped",
    "delta_mae",
    "MAE change when the group is dropped (>0: group helps)",
    err="se",
)
by_group

In [ ]:
by_feature = lab.ablation(
    data,
    lab.PREGAME_SAFE,
    {name: [name] for name in lab.PREGAME_SAFE},
    reference=reference,
)
barh(
    by_feature,
    "dropped",
    "delta_mae",
    "MAE change when the feature is dropped (>0: it helps)",
    err="se",
)
by_feature

## 7. Experiment: add and drop features

Experimental features are plain functions of the point-in-time `RushSnapshot`, the same input the real builder gets:

- `snapshot.prior_player_games` holds this player's earlier box scores only, oldest first. Each row has `kickoff_at`, `season`, `season_type`, `rushing_yards`, `carries`, `team` and `opponent_team`.
- `snapshot.prior_opponent_games` holds rushing yards the upcoming defense allowed in earlier games (in `rushing_yards`).
- `snapshot.target` is the schedule context for this game: `kickoff_at`, `season`, `week`, `player_team`, `home_team`, `away_team`, rest, `roof` and `surface`. Its `temp` and `wind` are game-day actuals, so don't use them.

Return `None` for unknown; it becomes NaN, never 0. `add_features` checks that no snapshot contains a game at or after kickoff.

Team-level inputs such as carry share or backfield competition aren't in the snapshot yet. Those need the assembler in `features/rush_yards_source.py` extended.

In [ ]:
def days_since_last_game(s):
    # Long gaps flag a player returning from injury or a benching.
    if not s.prior_player_games:
        return None
    return (s.target.kickoff_at - s.prior_player_games[-1].kickoff_at).days


def last2_carries_pg(s):
    games = s.prior_player_games[-2:]
    return None if not games else sum(g.carries for g in games) / len(games)


def career_games(s):
    return len(s.prior_player_games)


def last4_max_rush_yards(s):
    games = s.prior_player_games[-4:]
    return None if not games else max(g.rushing_yards for g in games)


lab.add_features(
    data,
    {
        "days_since_last_game": days_since_last_game,
        "last2_carries_pg": last2_carries_pg,
        "career_games": career_games,
        "last4_max_rush_yards": last4_max_rush_yards,
    },
)

Edit `DROP` and `ADD`, then rerun this cell. `delta_mae < 0` means the candidate is better. Compare `delta_mae` with `se` before believing it.

In [ ]:
DROP = {"div_game", "rest_days"}
ADD = ["days_since_last_game"]

candidate_features = [f for f in lab.PREGAME_SAFE if f not in DROP] + ADD
candidate = lab.evaluate(data, candidate_features)
result = lab.compare(data, reference, candidate)
row = result.filter(pl.col("season") == "overall").row(0, named=True)
print(
    f"{len(candidate_features)} features: MAE {row['mae_a']:.3f} -> {row['mae_b']:.3f} "
    f"(delta {row['delta_mae']:+.3f} +/- {2 * row['se']:.3f}), "
    f"RMSE {row['rmse_a']:.3f} -> {row['rmse_b']:.3f}"
)
result

To see how the candidate uses its features, rerun the importance views on it:

In [ ]:
lab.permutation_importance(data, candidate, repeats=3)

## 8. Holdout check (once, at the end)

Only after the feature list is final. If 2025 disagrees with the validation seasons, the gain was probably tuned to noise. Don't go back and tweak until 2025 agrees.

After that, promote the features into `features/rush_yards.py` as a new feature-set version (`rush_rb_v2`), and confirm with the real weekly backtest: `python -m models.rush_yards backtest`.

In [ ]:
RUN_HOLDOUT = False

if RUN_HOLDOUT:
    final_ref = lab.evaluate(
        data, lab.PREGAME_SAFE, seasons=[lab.HOLDOUT_SEASON], allow_holdout=True
    )
    final_new = lab.evaluate(
        data, candidate_features, seasons=[lab.HOLDOUT_SEASON], allow_holdout=True
    )
    display(lab.compare(data, final_ref, final_new))